# Notebook 5 — Mở rộng: FPS Benchmark, Visualization & Demo

**Bài tập lớn số 2 · CO5085 · HCMUT 2025-2026**

## Nội dung mở rộng (40% điểm)
1. FPS Benchmark chi tiết (batch sizes, image resolutions)
2. Confidence threshold sweep — tìm threshold tối ưu
3. GradCAM visualization cho Faster R-CNN backbone
4. Gradio web demo

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
from src.data import get_device, VOC_CLASSES, VOCDetectionDataset, get_val_transforms
from src.models import get_faster_rcnn
from src.train import load_frcnn_checkpoint
from src.evaluate import measure_fps
from src.utils import visualize_detections, load_metrics_json

device = get_device()
print("Device:", device)

## 1. FPS Benchmark: Chi tiết theo Image Size

In [ ]:
# Benchmark FPS với các image size khác nhau
image_sizes = [(320, 320), (480, 480), (640, 640), (800, 800)]

try:
    frcnn = get_faster_rcnn(num_classes=21)
    frcnn = load_frcnn_checkpoint(frcnn, '../results/checkpoints/frcnn_voc.pth', device)

    fps_results = {'Faster R-CNN': []}
    for h, w in image_sizes:
        fps_info = measure_fps(frcnn, 'frcnn', image_size=(h, w), n_runs=20, device=device)
        fps_results['Faster R-CNN'].append(fps_info['fps'])
        print(f"  {h}x{w}: {fps_info['fps']:.1f} FPS, {fps_info['ms_per_image']:.1f} ms")

    # Plot
    fig, ax = plt.subplots(figsize=(8, 4))
    sizes_str = [f"{h}x{w}" for h, w in image_sizes]
    for model_name, fps_list in fps_results.items():
        ax.plot(sizes_str, fps_list, 'o-', label=model_name, linewidth=2)
    ax.set_xlabel('Image Size')
    ax.set_ylabel('FPS')
    ax.set_title('FPS vs Image Size')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('../results/plots/fps_vs_size.png', dpi=120)
    plt.show()
except Exception as e:
    print(f"Cần có checkpoint: {e}")

## 2. Confidence Threshold Sweep

In [ ]:
# Ảnh hưởng của confidence threshold đến số lượng detections
try:
    from src.data import get_frcnn_loaders

    _, val_loader = get_frcnn_loaders('../data/voc', batch_size=4)

    thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    avg_detections = []

    frcnn.eval()
    images, _ = next(iter(val_loader))
    images = [img.to(device) for img in images]

    with torch.no_grad():
        outputs = frcnn(images)

    for thresh in thresholds:
        total = sum((o['scores'] >= thresh).sum().item() for o in outputs)
        avg_detections.append(total / len(images))

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(thresholds, avg_detections, 'o-', color='#6366f1', linewidth=2)
    ax.set_xlabel('Confidence Threshold')
    ax.set_ylabel('Avg Detections per Image')
    ax.set_title('Confidence Threshold vs Số lượng Detections (Faster R-CNN)')
    ax.grid(True, alpha=0.3)
    ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.7, label='threshold=0.5')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../results/plots/confidence_sweep.png', dpi=120)
    plt.show()
except Exception as e:
    print(f"Cần có checkpoint: {e}")

## 3. Feature Map Visualization

In [ ]:
# Visualize feature maps từ Faster R-CNN backbone (ResNet FPN)
try:
    activation = {}

    def hook_fn(name):
        def hook(module, input, output):
            activation[name] = output.detach()
        return hook

    # Đăng ký hook vào FPN
    frcnn.backbone.fpn.layer_blocks[-1].register_forward_hook(hook_fn('fpn_last'))

    ds = VOCDetectionDataset('../data/voc', year='2012', image_set='val',
                              transforms=get_val_transforms())
    img_t, target = ds[10]

    frcnn.eval()
    with torch.no_grad():
        _ = frcnn([img_t.to(device)])

    feat = activation.get('fpn_last')
    if feat is not None:
        feat_np = feat[0].cpu().numpy()

        fig, axes = plt.subplots(2, 4, figsize=(16, 8))
        axes = axes.flatten()
        for i, ax in enumerate(axes):
            if i < feat_np.shape[0]:
                ax.imshow(feat_np[i], cmap='viridis')
                ax.set_title(f'Channel {i}')
            ax.axis('off')
        plt.suptitle('FPN Feature Maps (last layer) — Faster R-CNN')
        plt.tight_layout()
        plt.savefig('../results/plots/feature_maps.png', dpi=100)
        plt.show()
    else:
        print("Hook không capture được feature map.")
except Exception as e:
    print(f"Feature map visualization: {e}")

## 4. Gradio Web Demo

In [ ]:
# Interactive demo với Gradio
# Chạy cell này để khởi động web interface

try:
    import gradio as gr
    from PIL import Image
    import torchvision.transforms as T
    import numpy as np, cv2

    model_loaded = False
    try:
        demo_model = get_faster_rcnn(num_classes=21)
        demo_model = load_frcnn_checkpoint(demo_model, '../results/checkpoints/frcnn_voc.pth', 'cpu')
        demo_model.eval()
        model_loaded = True
        print("Model loaded thành công!")
    except Exception as e:
        print(f"Không load được model: {e}")

    def detect_objects(pil_img, confidence_threshold=0.5):
        if not model_loaded:
            return pil_img, "Model chưa được load. Cần chạy training trước."

        transform = T.Compose([T.ToTensor(),
                                T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])])
        img_t = transform(pil_img)

        with torch.no_grad():
            output = demo_model([img_t])[0]

        keep = output['scores'] >= confidence_threshold
        boxes = output['boxes'][keep].numpy().tolist()
        labels = output['labels'][keep].numpy().tolist()
        scores = output['scores'][keep].numpy().tolist()

        # Vẽ boxes lên ảnh
        img_np = np.array(pil_img).copy()
        palette = [(255,50,50), (50,200,50), (50,50,255), (255,165,0), (128,0,128)]

        for box, label, score in zip(boxes, labels, scores):
            x1, y1, x2, y2 = map(int, box)
            color = palette[(label-1) % len(palette)]
            cv2.rectangle(img_np, (x1,y1), (x2,y2), color, 2)
            name = VOC_CLASSES[label-1] if 1 <= label <= 20 else f"cls{label}"
            cv2.putText(img_np, f"{name} {score:.2f}", (x1, max(y1-5,10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

        summary = f"Phát hiện {len(boxes)} đối tượng:\n"
        for box, label, score in zip(boxes, labels, scores):
            name = VOC_CLASSES[label-1] if 1 <= label <= 20 else f"cls{label}"
            summary += f"  {name}: {score:.2f}\n"

        return Image.fromarray(img_np), summary

    demo = gr.Interface(
        fn=detect_objects,
        inputs=[
            gr.Image(type="pil", label="Upload ảnh"),
            gr.Slider(0.1, 0.9, value=0.5, step=0.05, label="Confidence Threshold"),
        ],
        outputs=[
            gr.Image(type="pil", label="Kết quả Detection"),
            gr.Textbox(label="Danh sách đối tượng"),
        ],
        title="Object Detection Demo — Faster R-CNN trên Pascal VOC 2012",
        description="Upload ảnh để phát hiện 20 loại đối tượng (VOC classes)",
    )

    demo.launch(share=False, server_port=7860)
    print("Demo đang chạy tại http://localhost:7860")

except ImportError:
    print("Cần cài gradio: pip install gradio")
except Exception as e:
    print(f"Demo error: {e}")